In [23]:
import pandas as pd
from importlib import reload
import nbformat
import config
reload(config)

# Для корректного экспорта в HTML с интерактивными Plotly графиками
import plotly.io as pio
pio.renderers.default = "notebook_connected"


from config import (
    START_DATE, END_DATE, BACKTEST_START,
    INITIAL_INVESTMENT, CASH_RETURN_RATE, MANAGEMENT_FEE_MONTHLY,
    REBALANCE_FREQ, MOMENTUM_PERIODS, MOMENTUM_RANK, LOT_SIZE, TC_FIXED, TC_PCT,
    BENCHMARK_TICKERS_LIST, SPX_TICKER, VIX_TICKER, SPX_MA_PERIOD, VIX_THRESHOLD,
    REGIME_SPX_VIX_ENABLED, REGIME_DRAWDOWN_ENABLED, REGIME_VOLATILITY_ENABLED,
    DRAWDOWN_WINDOW, DRAWDOWN_THRESHOLD, DRAWDOWN_SOURCE, VOLATILITY_WINDOW,
    VOLATILITY_THRESHOLD, VOLATILITY_SOURCE, REGIME_FILTER_LOGIC, print_config_settings,
)

from data_loader import (
    fetch_price_data_divs,
    get_benchmark_prices,
    load_tickers,
)
from portfolio import build_client_portfolio_positions

from regime_filters import (
    CombinedRegimeFilter, DrawdownFilter, SpxVixFilter, VolatilityFilter,
)

from rolling_metrics import (
    calculate_all_rolling_metrics, calculate_expanding_metrics, 
    calculate_rolling_drawdown, calculate_rolling_mdd,
)

from strategy import (
    momentum_calculate_returns, momentum_get_rebalance_dates,
    momentum_hqm_table_calc, momentum_rank_score_calc,
    momentum_weights_table_calc,
)

from plotting import (
    plot_expanding_metrics, plot_filtered_backtest,
    plot_rolling_metrics, plot_strategy_equity,
    plot_strategy_metrics, plot_strategy_vs_benchmarks,
    print_filtered_comparison, print_strategy_vs_benchmarks_table
)

import execution_backtester

In [24]:
print_config_settings()

=== CONFIG SETTINGS ===
# Основные переменные
START_DATE: 2021-09-01
BACKTEST_START: 2023-01-01
END_DATE: 2026-02-25
INITIAL_INVESTMENT: 100000
BENCHMARK_TICKERS_LIST: ['SPY', 'SPLV', 'SPMO']
# Strategy parameters
REBALANCE_FREQ: W
MOMENTUM_PERIODS: [30, 90, 126]
MOMENTUM_RANK: 95
CASH_RETURN_RATE: 0.04
# Transaction cost parameters (for execution-based backtest)
TC_FIXED: 1.0
TC_PCT: 0.0005
LOT_SIZE: 1
MANAGEMENT_FEE_MONTHLY: 0.001

# =============================================================================

# REGIME FILTERS CONFIGURATION

# =============================================================================

# --- SPX/VIX Filter (market regime) ---
# Signal ON when: SPX > MA AND VIX < threshold
REGIME_SPX_VIX_ENABLED: True
SPX_TICKER: ^GSPC
VIX_TICKER: ^VIX
SPX_MA_PERIOD: 200
VIX_THRESHOLD: 25

# --- Drawdown Filter ---
# Signal ON when: rolling drawdown > threshold (less negative)
# Can be based on STRATEGY or any benchmark (SPY, SPMO, etc.)
REGIME_DRAWDOWN_ENABLED: Fa

## Выгрузка данных и формирование весов


In [3]:
file_name = 'SPX as of Apr 15 20251.xlsx'
tickers = load_tickers(file_name) 
price_table_wo_divs, price_table_with_divs, divs = fetch_price_data_divs(
                                                            tickers, 
                                                            start = START_DATE
                                                    )
index_price_table = get_benchmark_prices(
                            BENCHMARK_TICKERS_LIST, 
                            start=START_DATE
                    )

[**********************85%****************       ]  85 of 100 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
$HES: possibly delisted; no timezone found
[*********************100%***********************]  100 of 100 completed

1 Failed download:
['HES']: possibly delisted; no timezone found
[****                   8%                       ]  8 of 100 completed$IPG: possibly delisted; no price data found  (1d 2021-09-01 -> 2026-02-25) (Yahoo error = "No data found, symbol may be delisted")
[**********************46%                       ]  46 of 100 completed$K: possibly delisted; no price data found  (1d 2021-09-01 -> 2026-02-25) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  100 of 100 completed

2 Failed downloads:
['IPG', 'K']: possibly delisted; no price data found  (1d 2021-09-01 -> 2026-02-25) (Yahoo error = "No data found, symbol ma

In [41]:
# Рассчет доходностей по периодам (multi-period momentum)
momentum_returns = momentum_calculate_returns(
                        price_table_with_divs, 
                        periods=MOMENTUM_PERIODS
                    ) 
print(f"Momentum periods: {MOMENTUM_PERIODS}")

# Нахождение дат ребалансировок (uses first period from list)
momentum_rebalance_dates = momentum_get_rebalance_dates(
                                momentum_returns, 
                                freq=REBALANCE_FREQ
                            ) 

while pd.NaT in momentum_rebalance_dates:
    momentum_rebalance_dates.remove(pd.NaT)

# Нахождение перцентильного ранга в каждом из периодов по доходности
momentum_rank = momentum_rank_score_calc(
                    momentum_rebalance_dates,
                    momentum_returns
                ) 

# Рассчет общего перцентильного ранка за все периоды (averaging across all periods)
momentum_hqm = momentum_hqm_table_calc(
                    momentum_rank, 
                    momentum_returns,
                    rank=MOMENTUM_RANK
                ) 

# Рассчет весов
momentum_weights = momentum_weights_table_calc(
                        momentum_hqm, 
                        price_table_wo_divs, 
                        freq=REBALANCE_FREQ
                )

print(f"HQM selection: stocks with average rank > {MOMENTUM_RANK}% across {len(MOMENTUM_PERIODS)} periods")

Momentum periods: [30, 90, 126]
HQM selection: stocks with average rank > 95% across 3 periods


In [42]:
# Веса для дат ребалансировок
target_weights_rebal = momentum_weights.loc[momentum_rebalance_dates].copy()

## Backtest с учетом фильтров


In [43]:
combined_filter = CombinedRegimeFilter(logic=REGIME_FILTER_LOGIC)

if REGIME_DRAWDOWN_ENABLED:
    combined_filter.add_filter(DrawdownFilter(
        window=DRAWDOWN_WINDOW,
        threshold=DRAWDOWN_THRESHOLD,
        source=DRAWDOWN_SOURCE  # 'STRATEGY' or benchmark ticker like 'SPY'
    ))
    print(f"✓ Drawdown Filter: window={DRAWDOWN_WINDOW}d, threshold={DRAWDOWN_THRESHOLD:.0%}, source={DRAWDOWN_SOURCE}")

if REGIME_VOLATILITY_ENABLED:
    combined_filter.add_filter(VolatilityFilter(
        window=VOLATILITY_WINDOW,
        threshold=VOLATILITY_THRESHOLD,
        source=VOLATILITY_SOURCE
    ))
    print(f"✓ Volatility Filter: window={VOLATILITY_WINDOW}d, threshold={VOLATILITY_THRESHOLD:.0%}, source={VOLATILITY_SOURCE}")

if REGIME_SPX_VIX_ENABLED:
    spx_vix_filter = SpxVixFilter(
        enabled=True,
        spx_ma_period=SPX_MA_PERIOD,
        vix_threshold=VIX_THRESHOLD,
        spx_ticker=SPX_TICKER,
        vix_ticker=VIX_TICKER
    )

    spx_vix_filter.load_data(start_date=str(BACKTEST_START.date()), end_date=END_DATE)
    combined_filter.add_filter(spx_vix_filter)
    print(f"✓ SPX/VIX Filter: SPX>{SPX_MA_PERIOD}d MA and VIX<{VIX_THRESHOLD}")

print(f"\nFilter combination logic: '{REGIME_FILTER_LOGIC}' (OFF if {REGIME_FILTER_LOGIC} filter triggers)")
print(f"Total active filters: {len(combined_filter.filters)}")

✓ SPX/VIX Filter: SPX>200d MA and VIX<25

Filter combination logic: 'any' (OFF if any filter triggers)
Total active filters: 1


In [44]:

benchmark_prices_for_filters = index_price_table.copy()

(equity_filtered, returns_filtered, snapshots_filtered, trades_filtered) = execution_backtester.run_execution_backtest_with_filters(
    prices_df=price_table_with_divs,
    benchmark_prices_df=benchmark_prices_for_filters,
    target_weights_df=target_weights_rebal,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=LOT_SIZE,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY,
    regime_filter=combined_filter
)

print(f"\n✓ Filtered backtest complete!")
print(f"  Equity curve: {len(equity_filtered)} days")
print(f"  Total trades: {len(trades_filtered)}")
print(f"  Final value: ${equity_filtered.iloc[-1]:,.2f}")

Running execution backtest with dynamic filters...
  Start date: 2023-01-03
  End date: 2026-02-24
  Trading days: 788
  Rebalance dates: 165
  Initial capital: $100,000
  Active filters: ['SPX_VIX']
  Filter logic: any
  📉 2023-01-04: Regime OFF - liquidated 25 positions
  📉 2023-01-19: Regime OFF - liquidated 25 positions
  📉 2023-03-10: Regime OFF - liquidated 25 positions
  📉 2023-03-20: Regime OFF - liquidated 0 positions
  Processed 100/788 days, equity: $103,129, regime: ON
  Processed 200/788 days, equity: $107,172, regime: ON
  📉 2023-10-23: Regime OFF - liquidated 25 positions
  Processed 300/788 days, equity: $125,232, regime: ON
  Processed 400/788 days, equity: $113,879, regime: OFF
  Processed 500/788 days, equity: $131,139, regime: ON
  Processed 600/788 days, equity: $115,839, regime: ON
  Processed 700/788 days, equity: $143,498, regime: ON
  Processed 788/788 days, equity: $173,131, regime: ON

=== Backtest Complete ===
Final equity: $173,131.00
Total return: 73.13%
T

In [45]:
# Динамика портфеля
equity_filtered.head(5)

date
2023-01-03    99921.818466
2023-01-04    99758.165940
2023-01-05    99768.942156
2023-01-06    99779.719537
2023-01-09    99790.498082
Name: equity, dtype: float64

In [46]:
# Подневные доходности 
returns_filtered.head(5)

date
2023-01-03    0.000000
2023-01-04   -0.001638
2023-01-05    0.000108
2023-01-06    0.000108
2023-01-09    0.000108
Name: equity, dtype: float64

In [47]:
# Датафрем с данными по портфелю: 
# объем (в кэше/в рынке), стоимость позиций, комиссия, торгует ли стратегия
snapshots_filtered.head(5)

,equity,cash,holdings_value,num_positions,costs_today,mgmt_fee_today,regime_on,rolling_dd_30d,rolling_vol_30d,filter_SPX_VIX_on,filter_SPX_VIX_value
date,,,,,,,,,,,
2023-01-03,99921.818466,2344.390943,97577.427523,25,73.788714,4.758408,True,NaN,NaN,NaN,NaN
2023-01-04,99758.165940,99758.165940,0.000000,0,73.738369,4.750615,False,-0.001789,NaN,False,22.010000
2023-01-05,99768.942156,99768.942156,0.000000,0,0.000000,4.751128,False,-0.002418,0.012998,False,22.459999
2023-01-06,99779.719537,99779.719537,0.000000,0,0.000000,4.751641,False,-0.002311,0.015177,False,21.129999
2023-01-09,99790.498082,99790.498082,0.000000,0,0.000000,4.752155,False,-0.002203,0.014219,False,21.969999


In [48]:
# Сделки
trades_filtered.head(20)

,date,ticker,side,shares,price,notional,cost,reason
0,2023-01-03,ROST,BUY,36,112.883658,4063.811690,3.031906,REBALANCE
1,2023-01-03,WYNN,BUY,48,83.525491,4009.223555,3.004612,REBALANCE
2,2023-01-03,DECK,BUY,60,67.430000,4045.800018,3.022900,REBALANCE
3,2023-01-03,PCG,BUY,257,15.806727,4062.328766,3.031164,REBALANCE
4,2023-01-03,LVS,BUY,86,46.819106,4026.443095,3.013222,REBALANCE
5,2023-01-03,UHS,BUY,28,140.525262,3934.707327,2.967354,REBALANCE
6,2023-01-03,ACGL,BUY,68,59.535756,4048.431411,3.024216,REBALANCE
7,2023-01-03,BA,BUY,20,192.949997,3858.999939,2.929500,REBALANCE
8,2023-01-03,DHI,BUY,45,87.778240,3950.020822,2.975010,REBALANCE
9,2023-01-03,OMC,BUY,53,74.858909,3967.522154,2.983761,REBALANCE


In [49]:
# Динамика портфеля
fig = plot_strategy_equity(
    strategy_equity=equity_filtered,
    rebalance_dates=momentum_rebalance_dates,
    title="Strategy Equity Curve (Filtered)",
    snapshots_df=snapshots_filtered
)
fig.show()

In [50]:
# Метрики за все время существования стратегии
expanding_df = calculate_expanding_metrics(equity_filtered)

latest_date = expanding_df.index[-1]
cagr_latest = expanding_df['cagr'].iloc[-1] if not expanding_df.empty else None
vol_latest = expanding_df['vol'].iloc[-1] if not expanding_df.empty else None
sharpe_latest = expanding_df['sharpe'].iloc[-1] if not expanding_df.empty else None
mdd_latest = expanding_df['mdd'].iloc[-1] if not expanding_df.empty else None

print("=== Strategy Metrics (Latest, Expanding) ===")
print(f"Date: {latest_date.strftime('%Y-%m-%d')}")
print(f"CAGR: {(cagr_latest or 0):.2%}" if cagr_latest is not None else "CAGR: N/A")
print(f"Volatility (expanding): {(vol_latest or 0):.2%}" if vol_latest is not None else "Volatility (expanding): N/A")
print(f"Sharpe (expanding): {sharpe_latest:.2f}" if sharpe_latest is not None else "Sharpe (expanding): N/A")
print(f"MDD (to date): {(mdd_latest or 0):.2%}" if mdd_latest is not None else "MDD (to date): N/A")

# Plot expanding metrics (4x1 layout)
fig_expanding = plot_expanding_metrics(expanding_df, sharpe_ylim=4.0, cagr_ylim=0.8)
fig_expanding.show()

=== Strategy Metrics (Latest, Expanding) ===
Date: 2026-02-24
CAGR: 19.24%
Volatility (expanding): 20.40%
Sharpe (expanding): 0.96
MDD (to date): -16.52%


In [51]:
# Метрики за все 30 дневное окно 
ROLLING_WINDOW = 30

rolling_30d = calculate_all_rolling_metrics(equity_filtered, windows=[ROLLING_WINDOW])

# Latest values
dd_30d_latest = rolling_30d[f'dd_{ROLLING_WINDOW}d'].iloc[-1]
vol_30d_latest = rolling_30d[f'vol_{ROLLING_WINDOW}d'].iloc[-1]
sharpe_30d_latest = rolling_30d[f'sharpe_{ROLLING_WINDOW}d'].iloc[-1]

print(f"=== Rolling {ROLLING_WINDOW}-Day Metrics (Latest) ===")
print(f"Date: {rolling_30d.index[-1].strftime('%Y-%m-%d')}")
print(f"Drawdown ({ROLLING_WINDOW}d): {dd_30d_latest:.2%}")
print(f"Volatility ({ROLLING_WINDOW}d, ann.): {vol_30d_latest:.2%}")
print(f"Sharpe ({ROLLING_WINDOW}d): {sharpe_30d_latest:.2f}")

# Plot rolling 30-day metrics (3x1 layout)
fig_rolling = plot_rolling_metrics(
    rolling_df=rolling_30d,
    window=ROLLING_WINDOW,
    volatility_threshold=VOLATILITY_THRESHOLD,
    sharpe_ylim=6.0
)
fig_rolling.show()

=== Rolling 30-Day Metrics (Latest) ===
Date: 2026-02-24
Drawdown (30d): -0.39%
Volatility (30d, ann.): 32.63%
Sharpe (30d): 3.14


## Сравнение с бенчмарками

In [52]:
# Запуск бэктеста для бенчмарков по стратегии buy & hold
benchmark_results = execution_backtester.run_benchmark_backtest(
    benchmark_tickers=BENCHMARK_TICKERS_LIST,
    prices_df=index_price_table,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=LOT_SIZE,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    start_date=BACKTEST_START
)


print("\n=== Benchmark Summary ===")
for ticker, result in benchmark_results.items():
    print(f"\n{ticker}:")
    print(f"  Shares purchased: {result['shares']}")
    print(f"  Purchase price: ${result['first_price']:.2f}")
    print(f"  Transaction cost: ${result['cost']:.2f}")
    print(f"  Cash remaining: ${result['cash']:,.2f}")
    print(f"  Final value: ${result['final_value']:,.2f}")
    print(f"  Total return: {(result['final_value'] / INITIAL_INVESTMENT - 1)*100:.2f}%")


Running buy-and-hold benchmarks...
  Benchmarks: ['SPY', 'SPLV', 'SPMO']
  Initial capital: $100,000
  Transaction costs: $1.0 + 5.0 bps
  ✓ SPY: 270 shares @ $369.48, Final: $184,326.74 (84.33%)
  ✓ SPLV: 1671 shares @ $59.78, Final: $126,976.69 (26.98%)
  ✓ SPMO: 1815 shares @ $55.05, Final: $213,068.51 (113.07%)

=== Benchmark Summary ===

SPY:
  Shares purchased: 270
  Purchase price: $369.48
  Transaction cost: $50.88
  Cash remaining: $213.77
  Final value: $184,326.74
  Total return: 84.33%

SPLV:
  Shares purchased: 1671
  Purchase price: $59.78
  Transaction cost: $50.95
  Cash remaining: $64.25
  Final value: $126,976.69
  Total return: 26.98%

SPMO:
  Shares purchased: 1815
  Purchase price: $55.05
  Transaction cost: $50.96
  Cash remaining: $41.96
  Final value: $213,068.51
  Total return: 113.07%


In [53]:
# Сравнение стратегии с бенчмарками

fig = plot_strategy_vs_benchmarks(
    strategy_equity=equity_filtered,
    benchmark_results=benchmark_results,
    rebalance_dates=momentum_rebalance_dates,
    title="Strategy vs Benchmarks (Execution-Based with Transaction Costs)"
)
fig.show()


print_strategy_vs_benchmarks_table(
    strategy_equity=equity_filtered,
    benchmark_results=benchmark_results,
    trades_df=trades_filtered,
    initial_investment=INITIAL_INVESTMENT, 
    rolling_window=30
)


=== Performance Comparison (Since Inception, All with Transaction Costs) ===

Strategy             Final Value    Return    CAGR     MDD        Sharpe     Vol        Trades    
-----------------------------------------------------------------------------------------------
Momentum Strategy    $  173,131.00   73.13%  19.24%  -16.52%    0.96  20.40%    4099
SPY (B&H)            $  184,326.74   84.33%  21.65%  -19.74%    1.33  15.67%       1
SPLV (B&H)           $  126,976.69   26.98%   7.97%  -10.01%    0.74  11.25%       1
SPMO (B&H)           $  213,068.51  113.07%  27.43%  -23.27%    1.31  20.08%       1

=== Performance Comparison (30d Rolling, All with Transaction Costs) ===

Strategy             Final Value    Return    CAGR     MDD       Sharpe   Vol      Trades  
-----------------------------------------------------------------------------------------------
Momentum Strategy    $  173,131.00   12.27%  19.24%   -0.39%    3.14  32.63%    4099
SPY (B&H)            $  184,326.74   -

## Расчет позиций для клиентского портфеля

> ⚠️ **Важно:** Эту секцию нужно запускать ПОСЛЕ бэктеста с фильтрами!  
> Фильтры определяют, должна ли стратегия быть в рынке или в кэше.

In [54]:
CLIENT_PORTFOLIO_SIZE = 100000  # Размер портфеля клиента в USD
PORTFOLIO_DATE = None  # Необходимая дата (например, '2025-01-31') или None для последней доступной
SAVE_TO_EXCEL = True  # Сохранить ли в Excel в ./data

try:
    current_regime = snapshots_filtered['regime_on'].iloc[-1]
    regime_date = snapshots_filtered.index[-1]
    
    if not current_regime:
        print("⚠️  ВНИМАНИЕ: Фильтры указывают НА ВЫХОД В КЭШ!")
        print(f"   Дата: {regime_date.strftime('%Y-%m-%d')}")
        print(f"   Режим: {'INVESTED' if current_regime else 'CASH'}")
        print("\n   Рекомендация: НЕ открывать позиции, держать портфель в кэше.")
        print("   Позиции ниже показаны для информации (если бы фильтры были отключены).\n")
    else:
        print(f"✓ Фильтры: СТРАТЕГИЯ АКТИВНА (дата: {regime_date.strftime('%Y-%m-%d')})")
        print("  Можно открывать позиции.\n")

except NameError:
    print("⚠️  ВНИМАНИЕ: Сначала запустите бэктест с фильтрами (ячейки выше)!")
    print("   Без бэктеста невозможно определить текущее состояние фильтров.\n")

# Генерация позиций (для информации или для исполнения, если режим = INVESTED)
positions_df, summary, target_date, output_path = build_client_portfolio_positions(
    weights_df=momentum_weights,
    prices_df=price_table_wo_divs,
    portfolio_size=CLIENT_PORTFOLIO_SIZE,
    portfolio_date=PORTFOLIO_DATE,
    lot_size=LOT_SIZE,
    rounding_rule='floor',
    save_to_excel=SAVE_TO_EXCEL,
    output_dir='data',
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    verbose=True
)

positions_df.head(30)

✓ Фильтры: СТРАТЕГИЯ АКТИВНА (дата: 2026-02-24)
  Можно открывать позиции.

=== Client Portfolio Calculation ===
Target date: 2026-02-24
Portfolio size: $100,000
Saved positions to: data/2026-02-24_100000.xlsx

--- Position Summary ---
Number of positions: 25
Total invested (before costs): $97,415.13
Cash remaining (before costs): $2,584.87
Actual weight sum: 97.42%

--- Estimated Transaction Costs ---
Fixed costs (25 orders × $1.0): $25.00
Proportional costs (0.05% × $97,415): $48.71
Total estimated costs: $73.71

--- Net Summary ---
Net invested (with costs): $97,488.84
Net cash remaining: $2,511.16

List of positions:


,ticker,target_weight,price,shares,position_value,actual_weight
0,WDC,0.040998,281.940002,14,3947.160034,0.039472
1,TER,0.040915,323.019989,12,3876.239868,0.038762
2,STX,0.040832,410.109985,9,3690.989868,0.036910
3,GLW,0.040748,148.000000,27,3996.000000,0.039960
4,TPL,0.040665,503.700012,8,4029.600098,0.040296
5,MRNA,0.040582,50.189999,80,4015.199890,0.040152
6,AMAT,0.040499,376.589996,10,3765.899963,0.037659
7,CAT,0.040374,755.000000,5,3775.000000,0.037750
8,FDX,0.040374,383.709991,10,3837.099915,0.038371
9,MU,0.040249,429.269989,9,3863.429901,0.038634


## Сравнение стратегии с фильтрами/без фильтров

In [55]:
print(f"Target weights shape: {target_weights_rebal.shape}")
print(f"Rebalance dates: {len(momentum_rebalance_dates)}")
print(f"First rebalance date: {target_weights_rebal.index[0]}")
print(f"Last rebalance date: {target_weights_rebal.index[-1]}")

# Run execution backtest
equity_curve_exec_no_filters, daily_returns_exec_no_filters, snapshots_df_no_filters, trades_df_no_filters = execution_backtester.run_execution_backtest(
    prices_df=price_table_with_divs,
    target_weights_df=target_weights_rebal,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    lot_size=LOT_SIZE,
    cash_return_rate=CASH_RETURN_RATE,
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY
)

print(f"\nExecution backtest complete!")
print(f"Equity curve shape: {equity_curve_exec_no_filters.shape}")
print(f"Daily returns shape: {daily_returns_exec_no_filters.shape}")
print(f"Snapshots shape: {snapshots_df_no_filters.shape}")
print(f"Trades shape: {trades_df_no_filters.shape}")

Target weights shape: (235, 503)
Rebalance dates: 235
First rebalance date: 2021-09-01 00:00:00
Last rebalance date: 2026-02-23 00:00:00
Running execution backtest...
  Start date: 2023-01-03
  End date: 2026-02-24
  Trading days: 788
  Rebalance dates: 165
  Initial capital: $100,000
  Transaction costs: $1.0 + 5.0 bps
  Management fee: 0.10% monthly (1.20% annual)
  Processed 100/788 days, equity: $111,067
  Processed 200/788 days, equity: $115,340
  Processed 300/788 days, equity: $137,129
  Processed 400/788 days, equity: $124,820
  Processed 500/788 days, equity: $147,335
  Processed 600/788 days, equity: $134,428
  Processed 700/788 days, equity: $167,013
  Processed 788/788 days, equity: $208,513

=== Backtest Complete ===
Final equity: $208,513.49
Total return: 108.51%
Total trades: 4332
Total transaction costs: $11,076.57
Total management fees: $5,087.77
Total all costs: $16,164.33
Cash never went negative: ✓

Execution backtest complete!
Equity curve shape: (788,)
Daily retur

In [56]:
fig = plot_filtered_backtest(
    equity_unfiltered=equity_curve_exec_no_filters,
    equity_filtered=equity_filtered,
    snapshots_filtered=snapshots_filtered,
    rebalance_dates=momentum_rebalance_dates,
    drawdown_threshold=DRAWDOWN_THRESHOLD,
    initial_investment=INITIAL_INVESTMENT,
    title="Equity Curves: With vs Without Dynamic Regime Filters"
)
fig.show()


print_filtered_comparison(
    equity_unfiltered=equity_curve_exec_no_filters,
    equity_filtered=equity_filtered,
    initial_investment=INITIAL_INVESTMENT
)


=== Performance Comparison ===
Without filters: $208,513.49 (108.51%)
With filters:    $173,131.00 (73.13%)
Difference:      $-35,382.49
